# nano Beatrix v1.1 — the frames-then-carry arm

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/geolip-nano-beatrix-trainer/blob/main/colab/nano_v1_1_frames_carry.ipynb)

The carry arms (2026-10-17) showed that a v1.1 craft continued on carry rows learns to write its own carries and answers a question about 3,900 bytes past the only frame in 0.28 to 0.59 of rows, where the frames arm and the control answer none. They also showed that the two data forms train two skills: the carry arms' near reads stayed at copy 0.29 to 0.44 while the frames arms' reached 0.89 to 0.93. This notebook trains the arm that puts the two in one body: THE FRAMES-THEN-CARRY ARM starts from the frames arm's end state (the exact state: fp32 weights and both optimizers at step 65,237) and runs the carry phase (1.0 billion bytes, 3,815 steps; the same mix and dose as the carry arm, in a different shuffle because the phase sits one slot later in the plan).

| craft (one a body) | starts from | its phase |
|---|---|---|
| nano-<body>-v1.1-fr-cf | the frames arm's end state (step 65,237; copy 0.89 to 0.93) | three rows in four carry rows, the rest plain fineweb-edu text; 1.0 billion bytes, steps 65,237 to 69,052 |

The reads: the before row (the frames end, the same numbers as the frames arm's last row), the bed's frame gauges and the carry gauge at 66,000, 68,000 and the end, the full eval every 2,000 steps. C4 tabulates the frames end, the end of this arm, the carry arm and the control, and runs the carry gauge on this arm's end beside the carry arm's end.

The pass rule, written before the run: NEAR KEPT when this arm's copy stays within 0.10 of the frames end; FAR GAINED when its far question whole reaches 0.20 or more (the frames end reads 0.00); TEXT when its held-out bits sit within 0.01 of the carry arm's. NEAR and FAR both met: one body holds both skills. NEAR lost while FAR is gained: the carry phase displaces the near skill, and a mixed phase is the next arm. FAR not gained: a frames body does not form the habit in 1.0 billion bytes from this start.

Length: about 27 minutes a body on the card the frames train ran on, plus the before reads and the carry gauge (a few minutes a read). One body by default (Pollux dual: the best carry body at 0.59 and the cheapest step); the last cell shows how to run the others. Needs: the frames arm's end state in the repo (C2 checks), and HF_TOKEN in this notebook's secrets with write access to AbstractPhil/alephllm-nano-castor-pollux.

In [ ]:
# C1 — pinned installs + THE DECISION BLOCK. RESTART RUNTIME if the version line below changes.
import os, sys
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'   # a progress bar is not allowed to cost a session
%pip install -q "geolip-nano-beatrix-trainer[corpus] @ git+https://github.com/AbstractEyes/geolip-nano-beatrix-trainer@v0.1.49"
import torch
import geolip.nano_beatrix as N
import geolip.alephllm as A
from geolip.nano_beatrix import nano_v1 as V1
print('geolip.nano_beatrix', N.__version__, '| geolip.alephllm', A.__version__, '| torch', torch.__version__,
      '| card', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert tuple(int(x) for x in N.__version__.split('.')) >= (0, 1, 49), (
    'RESTART REQUIRED: the runtime still holds an older package — Runtime > Restart, then rerun from C1')
assert torch.cuda.is_available(), 'no card: Runtime > Change runtime type > GPU'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception:
        return os.environ.get('HF_TOKEN')
HF_TOKEN = _token()                                   # the value is never printed
assert HF_TOKEN, 'HF_TOKEN missing: the start states are read from the repo and the continuations ship to it'
os.environ['HF_TOKEN'] = HF_TOKEN

# ============ THE DECISION BLOCK — every setting from the record, with its source beside it ============
OUT = '/content/nano_v1_1_frames_carry'                     # the session folder: the corpus, the runs, the bed's gauge folder
REPO = V1.REPO                                        # AbstractPhil/alephllm-nano-castor-pollux: the v1.1 folders read, the new ones written
NAMES = ['nano-castor-v1.1-fr-cf', 'nano-pollux-canon-v1.1-fr-cf', 'nano-pollux-dual-canon-v1.1-fr-cf']   # 2026-10-18: the three bodies not yet run (Pollux dual ran 10-18: far gained, near lost); a craft already in the repo resumes
MB, GA = 16, 4                                        # the 2s shape, the fastest on the v1.1 benchmark (262,144 bytes a step)
GAUGE_EVERY = 2000                                    # the bed's gauges at 62,000 and 64,000 (every full eval) and at the end
CORPUS_BYTES = 1.0e9                                  # the frames corpus (fineweb-edu, the held-out head skipped), built once in C2
MAX_HOURS = None                                      # this session's budget in hours (None: until every craft ends or the cell is stopped)
print('decision block:', len(NAMES), 'crafts:', ', '.join(NAMES), '| the phase', V1.CONT_TOKENS / 1e9, 'B bytes |', MB, 'x', GA,
      '| gauges every', GAUGE_EVERY, 'steps | corpus', CORPUS_BYTES / 1e9, 'GB | ships to', REPO)

In [ ]:
# C2 (P) — PREFLIGHT: every frames arm's end state in the repo, the held-out head, the frames corpus (built once; about a minute or two
# a gigabyte from the stream), the rows' composition pace against the card's, the push probe. Nothing trains here.
import json, time
from huggingface_hub import HfApi
from geolip.nano_beatrix.bed_corpus import write_heldout, build as build_corpus
from geolip.nano_beatrix import frames_data as FD
api = HfApi(token=HF_TOKEN)
files = set(api.list_repo_files(REPO))
missing = []
for n in NAMES:
    spec = V1.cont_init(n, MB * GA * V1.CTX)
    path = f"{spec['prefix']}/{spec['payload']}"
    if path not in files:
        missing.append(path)
assert not missing, f'not every frames arm has its end state: missing {sorted(set(missing))}'
print('start states: every frames arm has its end state at step', spec['step'])
held = write_heldout(f'{OUT}/bed')
folder = f'{OUT}/frames_corpus'
if not os.path.exists(f'{folder}/bed_corpus.bin'):
    build_corpus(folder, int(CORPUS_BYTES))
packed = FD.load_corpus(folder)
n, t0 = 0, time.time()
while time.time() - t0 < 10:
    FD.row_bytes(packed, 1337, n, FD.P_FRAMED, FD.CARRY, carry=True)
    n += 1
rate = n / (time.time() - t0)
need = 879_000 / FD.ROW_BYTES                         # the fastest craft's rows a second (Castor, 879k bytes a second)
print(f'frames corpus {len(packed) / 1e9:.2f} GB; the frames rows compose at {rate:,.0f} a second against {need:.0f} needed'
      + ('' if rate > 1.5 * need else ' — TOO SLOW: the card will wait on the data'))

PUSH_OK = False
path = 'preflight/push_probe_v1_1_frames_carry.json' # its own probe file
body = json.dumps({'utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'card': torch.cuda.get_device_name(0),
                   'compose_rows_s': rate, 'crafts': NAMES}).encode()
api.upload_file(path_or_fileobj=body, path_in_repo=path, repo_id=REPO, commit_message='nano v1.1 frames-then-carry: push probe')
PUSH_OK = path in set(api.list_repo_files(REPO))
print('push probe', 'READ BACK OK' if PUSH_OK else 'NOT VISIBLE after upload', path)

# the gauge's smoke on this card before any training: the per-row dump on 8 rows of the start state (about a minute; a frames body
# writes no carries, so the far question reads 0.00 here)
src = V1.ALL[NAMES[0]]['source']
smoke = V1.gauge_stored(src, spec['step'], OUT, repo=REPO, carry=True, token=HF_TOKEN, extra=dict(carry_rows=8, carry_dump=1))
c = smoke['carry']
rows8 = sum(1 for _ in open(f"{OUT}/bed/{c['dump']['jsonl']}", encoding='utf-8'))
assert rows8 == c['n'] == 8, (rows8, c['n'])
print(f"gauge smoke on {src} at {spec['step']:,}: {rows8} rows dumped; carries a row {c['carries_per_row']:.2f}; "
      f"far question whole {c['full_value']:.2f}")

In [ ]:
# C3 (T) — THE CONTINUATIONS: each craft starts from its v1.1 craft's final state (its first row: the before, read by the gauges), then
# runs its phase; the trainer's lines and the bed's gauge lines ([v1]) print as they come. Rerun the cell to continue (a new session
# too). The cell's stop button saves the running craft's state before it stops.
assert PUSH_OK, 'the push probe did not read back: nothing would ship'
HIST = V1.run(OUT, HF_TOKEN, NAMES, micro_batch=MB, grad_accum=GA, repo=REPO, gauge_every=GAUGE_EVERY, max_hours=MAX_HOURS,
              frames_corpus=f'{OUT}/frames_corpus')
from IPython.display import Markdown, display
display(Markdown(V1.cont_table(HIST)))

In [ ]:
# C4 — THE FOUR-COLUMN READ AND READING WITH NOTES: the boundary rows of this body's four arms from the repo (frames, control, carry,
# frames-then-carry), the table (the frames end = the start, after, the carry arm, the control, the differences), then the carry gauge on
# the frames-then-carry craft's end beside the carry arm's end (192 rows each; gauge_stored pulls the final and runs the gauge here, a
# few minutes a read). Run any time; it trains nothing.
import json
from IPython.display import Markdown, display
from huggingface_hub import snapshot_download
BODIES = [n[:-6] for n in NAMES]                       # nano-<body>-v1.1
ALL_ARMS = [f'{b}-{a}' for b in BODIES for a in ('fr', 'ct', 'cf', 'fr-cf')]
hub = snapshot_download(REPO, token=HF_TOKEN, local_dir=f'{OUT}/hub',
                        allow_patterns=[f'{n}/runs/*' for n in ALL_ARMS] + [f'{n}/v1_boundaries.json' for n in ALL_ARMS])
hist = {}
for n in ALL_ARMS:
    p = f'{hub}/{n}/v1_boundaries.json'
    hist[n] = json.load(open(p, encoding='utf-8')) if os.path.exists(p) else []
display(Markdown(V1.cont2_table(hist)))
TPS = MB * GA * V1.CTX
rows = ['| body | craft | end step | whole value at the far question, own carries | first byte | value bits | carries a row | rows with a carry | exact copies | open probability at the grid | distance to the latest definition |',
        '|---|---|---|---|---|---|---|---|---|---|---|']
for b in BODIES:
    for arm in ('fr-cf', 'cf'):
        n = f'{b}-{arm}'
        end = V1.plan_steps(n, TPS)                      # 69,052 for the frames-then-carry craft, 65,237 for the carry arm
        if not any(e.get('phase') == V1.CONT_PHASE['cf'][0] for e in hist.get(n, [])):
            rows.append(f'| {b} | {arm} | {end:,} | not finished | | | | | | | |')
            continue
        c = V1.gauge_stored(n, end, OUT, repo=REPO, carry=True, token=HF_TOKEN).get('carry') or {}
        rows.append(f"| {b} | {arm} | {end:,} | {c.get('full_value', float('nan')):.3f} | {c.get('first_byte', float('nan')):.3f} | {c.get('value_bits', float('nan')):.1f} | "
                    f"{c.get('carries_per_row', float('nan')):.2f} | {c.get('rows_with_carries', float('nan')):.2f} | {c.get('carry_exact', float('nan')):.2f} | "
                    f"{c.get('open_prob', float('nan')):.3f} | {c.get('dist_latest', float('nan')):.0f} |")
display(Markdown('\n'.join(rows)))
display(Markdown(V1.table(hist)))
%load_ext tensorboard
%tensorboard --logdir {hub}

### Changing the run

Edit the decision block in C1, rerun C1 (the install is cached), then C2 and C3. A craft already in the repo resumes; it never restarts.

| to | the edit in C1 |
|---|---|
| all four bodies | `NAMES = list(V1.CRAFTS_CONT2)` (Pollux dual canon, Pollux canon, Pollux dual, Castor, in that order; about two hours) |
| another single body | `NAMES = ['nano-castor-v1.1-fr-cf']`, or `nano-pollux-canon-v1.1-fr-cf`, or `nano-pollux-dual-canon-v1.1-fr-cf` |
| a session budget | `MAX_HOURS = 1` |
| two cards | the first card `NAMES = list(V1.CRAFTS_CONT2)[:2]`, the second `NAMES = list(V1.CRAFTS_CONT2)[2:]` |

The session folder is the Colab disk: the repo holds the records.